In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv("data/processed/telco_clean.csv")

In [3]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,0.0,DSL,0,...,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,0.0,DSL,1,...,1,0,0,0,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,0.0,DSL,1,...,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,0.0,DSL,1,...,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,0.0,Fiber optic,0,...,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,1,1,24,1,1.0,DSL,1,...,1,1,1,1,One year,1,Mailed check,84.80,1990.50,0
7039,2234-XADUH,Female,0,1,1,72,1,1.0,Fiber optic,0,...,1,0,1,1,One year,1,Credit card (automatic),103.20,7362.90,0
7040,4801-JZAZL,Female,0,1,1,11,0,0.0,DSL,1,...,0,0,0,0,Month-to-month,1,Electronic check,29.60,346.45,0
7041,8361-LTMKD,Male,1,1,0,4,1,1.0,Fiber optic,0,...,0,0,0,0,Month-to-month,1,Mailed check,74.40,306.60,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   int64  
 4   Dependents        7043 non-null   int64  
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   int64  
 7   MultipleLines     7043 non-null   float64
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   int64  
 10  OnlineBackup      7043 non-null   int64  
 11  DeviceProtection  7043 non-null   int64  
 12  TechSupport       7043 non-null   int64  
 13  StreamingTV       7043 non-null   int64  
 14  StreamingMovies   7043 non-null   int64  
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   int64  


In [5]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

df["num_addons"] = df[addon_cols].sum(axis = 1)
df["is_first_year"] = (df["tenure"] <= 12).astype (int)
df["monthly_echeck"] = ((df["Contract"] == "Month-to-month") & (df["PaymentMethod"] == "Electronic check")).astype(int)
df["lives_alone"] = ((df["Partner"] == 0) & (df["Dependents"] == 0)).astype(int)
df[["num_addons", "is_first_year", "monthly_echeck", "lives_alone"]].head()

,num_addons,is_first_year,monthly_echeck,lives_alone
0,1,1,1,0
1,2,0,0,1
2,2,1,0,1
3,3,0,0,1
4,0,1,1,1


In [6]:
for col in ["is_first_year", "monthly_echeck", "lives_alone"]:
    print(df.groupby(col)["Churn"].mean().mul(100).round(1))
    

is_first_year
0    17.1
1    47.4
Name: Churn, dtype: float64
monthly_echeck
0    16.8
1    53.7
Name: Churn, dtype: float64
lives_alone
0    19.8
1    34.2
Name: Churn, dtype: float64


In [7]:
df.groupby("num_addons")["Churn"].mean().mul(100).round(1)

num_addons
0    21.4
1    45.8
2    35.8
3    27.4
4    22.3
5    12.4
6     5.3
Name: Churn, dtype: float64

In [8]:
df[df["InternetService"] != "No"].groupby("num_addons")["Churn"].mean().mul(100).round(2)

num_addons
0    52.24
1    45.76
2    35.82
3    27.37
4    22.30
5    12.43
6     5.28
Name: Churn, dtype: float64

In [9]:
text_cols = ["gender", "InternetService", "Contract", "PaymentMethod"]

X = df.drop(columns = ["customerID", "Churn"])
X = pd.get_dummies(X, columns = text_cols, drop_first = True, dtype = int)
Y = df["Churn"]

In [10]:
print(X.shape)
X.columns.tolist()

(7043, 27)


['SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'num_addons',
 'is_first_year',
 'monthly_echeck',
 'lives_alone',
 'gender_Male',
 'InternetService_Fiber optic',
 'InternetService_No',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check']

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.2, stratify = Y, random_state = 42)

print("Train :", X_train.shape, "Test :", X_test.shape)
print("Churn rate in train :", round(Y_train.mean(), 4))
print("Churn rate in test :", round(Y_test.mean(), 4))

Train : (5634, 27) Test : (1409, 27)
Churn rate in train : 0.2654
Churn rate in test : 0.2654


In [12]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "num_addons"]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [13]:
print(X_train_scaled[num_cols].describe().loc[["mean", "std"]].round(2))


      tenure  MonthlyCharges  TotalCharges  num_addons
mean    -0.0            -0.0           0.0         0.0
std      1.0             1.0           1.0         1.0


In [14]:
print(X_test_scaled[num_cols].describe().loc[["mean", "std"]].round(2))

      tenure  MonthlyCharges  TotalCharges  num_addons
mean   -0.02           -0.03         -0.04       -0.05
std     1.00            0.99          0.97        0.98


In [15]:
print("Churn inside X_train :", "Churn" in X_train.columns)
print("customerID inside X_train :", "customerID" in X_train.columns)
print("Customers in BOTH train and test :", len(set(X_train.index) & set(X_test.index)))
print("Train + test rows :", len(X_train) + len(X_test), "| original rows :", len(df))
print("Missing values in X_train :", X_train.isnull().sum().sum(), "| X_test :", X_test.isnull().sum().sum())

Churn inside X_train : False
customerID inside X_train : False
Customers in BOTH train and test : 0
Train + test rows : 7043 | original rows : 7043
Missing values in X_train : 0 | X_test : 0


In [16]:
X_train.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/X_train.csv", index = False)

X_test.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/X_test.csv", index = False)

X_train_scaled.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/X_train_scaled.csv", index = False)

X_test_scaled.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/X_test_scaled.csv", index = False)

Y_train.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/Y_train.csv", index = False)

Y_test.to_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/Y_test.csv", index = False)


In [17]:
print(pd.read_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/X_train.csv").shape)

(5634, 27)


In [18]:
print(pd.read_csv("C:/Users/slmkc/Downloads/PhaseI/Churn_model/data/processed/Y_train.csv").shape)

(5634, 1)


In [19]:
feature_notes = """ Feature notes -- IBM Telco churn
# New feature - built only from input columns, never from churn
# is_first_year: 1 if tenure <= 12 months. EDA : 47.4% churn in the first year
# monthly_echeck : 1 if month to month contract AND electronic check 53.7 % churn
# lives_alone : 1 if no partner AND no dependents 34.2 % churn vs 19.8 %

## Encoding:
# gender, InternetService, Contract, PaymentMethod turned into 0/1 (one-hot, first category dropped).

## Split and Scaling:
# 80% train / 20% test, random_state 42, Test set stays locked until the final evaluation
# tenure, MonthlyCharges, TotalCharges, num_addons scaled with StandardScaler fitted on the training set only

## customerID and Churn are not in X
# No customer appears in both train and test
# The scaler learned only from training data
# Limitation : the effect is small for simple flags, but next time explore the training set only
"""

with open("C:/Users/slmkc/Downloads/PhaseI/Churn_model/reports/feature_notes.md", "w") as f:
    f.write(feature_notes)